# Router Before the LLM: submission notebook

A cost-aware intent router for a personal finance assistant. This notebook is thin on purpose: it imports from `src/router/` and calls functions, so the same code runs here, in `scripts/` and in the demo.

**How to run:** Restart and Run All. With `FAST = True` (the default) it needs no GPU, no API key and no paid call. Numbers computed in fast mode are labelled as fast-mode numbers. The reported results are loaded from `results/`, where the scripts wrote them.

Sections follow the marking rubric: problem, data and preprocessing, models, evaluation, router and cost, limitations.

In [ ]:
# Cell 1: environment.
# On Colab: clone the repo and install the pinned packages. Locally: nothing to install.
import os
import sys
from importlib.metadata import version

IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""  # set once the GitHub repo exists (needed for the Colab clean-run check)

if IN_COLAB:
    import subprocess
    if not REPO_URL:
        raise RuntimeError("REPO_URL is not set yet, so the repo cannot be cloned on Colab.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "router-project"], check=True)
    os.chdir("router-project")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    sys.path.insert(0, os.path.abspath("src"))  # make `import router` work without a restart

print("Python", sys.version.split()[0])
for pkg in ["torch", "transformers", "datasets", "sentence-transformers", "scikit-learn", "pandas", "cleanlab"]:
    print(f"{pkg:>22} {version(pkg)}")

In [ ]:
# Cell 2: run mode and seeds.
FAST = True  # True: quick run, no GPU or API key needed. False: re-run the full five-seed scripts (Apple M3 Pro GPU, measured 22 Nov 2026: S1 30 min; E2 retrains S1 and S3 on cleaned labels in another 47 min).
SEED = 42

import time

from router.seed import pick_device, set_seed

set_seed(SEED)
NOTEBOOK_START = time.perf_counter()
print("FAST =", FAST, "| SEED =", SEED, "| device:", pick_device())

## 1. Problem

A finance assistant sends every user query to an LLM, which costs money and time on each call. A small local model (the router) looks at each query first and either handles it directly, rejects it as out of scope, or escalates it to the LLM when unsure.

**Research question:** how many LLM calls can a small router safely remove, and at what error cost? The headline metric is coverage at matched risk: the largest share of queries the router can decide itself while its error rate stays at or below the LLM's.

## 2. Data and preprocessing

In-scope queries come from **Banking77** (Casanueva et al., 2020): 77 banking intents. Out-of-scope queries come from **CLINC150** (Larson et al., 2019). Both are loaded at a pinned Hub revision. Splits are created once by `scripts/make_splits.py` and saved to `data/processed/`, so every system sees the same data.

Pipeline (`src/router/preprocess.py`): unicode NFKC normalisation and whitespace cleanup, removal of empty rows and duplicates, a train/test leakage check, a stratified 10% validation split, token length checks and class balance. The full record is in `results/data_report.md`.

In [ ]:
import json

import pandas as pd

from router import data

splits = {name: data.load_split(name) for name in data.SPLIT_FILES}
pd.DataFrame({"rows": {name: len(df) for name, df in splits.items()}})

In [ ]:
# Rows removed at each step, per source (every removed row is listed in removed_rows.csv)
removed = pd.read_csv(data.PROCESSED_DIR / data.REMOVED_ROWS_FILE, keep_default_na=False)
removed.groupby(["source", "reason"]).size().unstack(fill_value=0)

In [ ]:
# Token lengths per tokeniser, measured on train. max_length is used by S1 (and later S2, S3).
with open(data.PROCESSED_DIR / "token_lengths.json") as f:
    lengths = json.load(f)
pd.DataFrame(
    {
        name: {
            "p50": v["single"]["p50"],
            "p99": v["single"]["p99"],
            "max": v["single"]["max"],
            "max_length": v["single"]["max_length"],
            "test truncated": f"{v['truncated_share']['test']:.2%}",
            "pair max_length (S3)": v["pair"]["max_length"],
        }
        for name, v in lengths.items()
    }
).T

In [ ]:
# Class balance: queries per intent in train. Uneven counts are why macro averages are primary.
import matplotlib.pyplot as plt

counts = splits["train"]["label"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(len(counts)), counts.values)
ax.set_xlabel("intent (sorted by count)")
ax.set_ylabel("train queries")
ax.set_title(f"Banking77 train: {counts.min()} to {counts.max()} queries per intent")
plt.show()

## 3. Models

Every system returns the same three things per query: a predicted intent, an in-scope confidence and an out-of-scope score (1 minus that confidence). That shared interface (`src/router/systems/base.py`, and `registry.load_predictor`, which turns any trained system into one function from texts to predictions) is what makes the comparison fair.

- **S0, TF-IDF + logistic regression** (`s0_tfidf.py`): the cheapest possible baseline. It shows how much of the task keywords alone can solve. Its n-gram range and regularisation strength were chosen on validation macro-F1 (`results/s0_grid.json`).
- **S1, fine-tuned DistilRoBERTa** (`s1_encoder.py`): the standard approach for this topic. A small pretrained encoder with a softmax head, fine-tuned for 5 epochs with fixed, standard hyperparameters. Confidence is the maximum softmax probability. DeBERTa-v3-small was the other candidate but could not be trained on the available hardware (`results/s1_selection.json`).
- **S2, embedding nearest neighbour** (`s2_embedding.py`): a frozen sentence-embedding model (all-MiniLM-L6-v2, chosen over MPNet in `results/s2_selection.json`). Each intent scores the cosine similarity of its closest train example; that similarity is the confidence. No training, and a new intent is added by adding examples. It is also S3's retriever.
- **S3, pairwise paraphrase matcher** (`s3_pairwise.py`), the main system: a cross-encoder asks "do these two queries mean the same thing?" for the query and each of 20 examples shortlisted by S2, and the best pair wins. Confidence is that pair's probability, and the matched example explains the decision. It starts from a paraphrase model (Quora question pairs) and is fine-tuned on pairs from the train split.

S4, the LLM the router is meant to call less often, is described in section 5.

In [ ]:
# FAST = False: re-run the same scripts that produced the reported results.
import subprocess

if not FAST:
    for command in [
        ["scripts/train.py", "--system", "s0"],
        ["scripts/train.py", "--system", "s1"],
        ["scripts/train.py", "--system", "s2"],
        ["scripts/train.py", "--system", "s3"],
        ["scripts/evaluate.py", "--split", "val"],
        ["scripts/predict_format_control.py", "--split", "val"],
        ["scripts/run_s4.py", "--split", "val"],  # reads the committed S4 cache: no API call, no AWS account needed
        ["scripts/run_s4.py", "--split", "val", "--config", "s4_llm_fewshot"],
        ["scripts/evaluate.py", "--split", "val_fmt"],
        ["scripts/evaluate_oos.py", "--split", "val"],
        ["scripts/evaluate_oos.py", "--split", "val_fmt"],
        ["scripts/evaluate_router.py", "--split", "val_fmt"],
        # E2 (section 4c): flag label issues, retrain every system without them, compare.
        ["scripts/find_label_issues.py"],  # never replaces an audit sheet the owner has started
        ["scripts/train.py", "--system", "s0", "--labels", "cleaned"],
        ["scripts/train.py", "--system", "s1", "--labels", "cleaned"],
        ["scripts/train.py", "--system", "s2", "--labels", "cleaned"],
        ["scripts/train.py", "--system", "s3", "--labels", "cleaned"],
        ["scripts/evaluate_label_noise.py", "--split", "val"],
    ]:
        subprocess.run([sys.executable, *command], cwd=data.PROJECT_ROOT, check=True)
else:
    print("FAST mode: skipping the full five-seed re-run; reported results are loaded from results/.")

In [ ]:
# S0: train with the parameters chosen on validation, then predict the validation set.
from router.systems import s0_tfidf, s1_encoder
from router.systems.base import label_names_in_id_order

label_names = label_names_in_id_order()
train_df = splits["train"]
val_in_scope = splits["val"]  # E1 scores in-scope queries only

s0_cfg = data.load_config("s0_tfidf")
with open(data.RESULTS_DIR / "s0_grid.json") as f:
    s0_params = json.load(f)["chosen"]

start = time.perf_counter()
s0_model = s0_tfidf.train(train_df, s0_params, s0_cfg, seed=s0_cfg["seeds"][0])
s0_preds = s0_tfidf.predict(s0_model, val_in_scope["text"].tolist(), label_names)
print(f"S0 {s0_params}: trained and predicted in {time.perf_counter() - start:.1f}s")

In [ ]:
# S1: load the saved seed-0 checkpoint if this machine has it; otherwise train briefly (fast mode).
s1_cfg = data.load_config("s1_encoder")
checkpoint = s1_cfg["checkpoint"]
device = pick_device()
max_length = s1_encoder.max_length_for(checkpoint)
saved = data.PROJECT_ROOT / "checkpoints" / "s1" / s1_encoder.system_tag(checkpoint) / "seed0"

start = time.perf_counter()
if saved.exists():
    s1_model, s1_tokenizer = s1_encoder.load(saved, device)
    s1_source = "saved seed-0 checkpoint"
else:
    steps = s1_cfg["fast_steps"]
    s1_model, s1_tokenizer, _ = s1_encoder.train(train_df, s1_cfg, checkpoint, seed=0, device=device, label_names=label_names, max_steps=steps)
    s1_source = f"trained here for {steps} steps only (no checkpoint on this machine)"
s1_preds = s1_encoder.predict(s1_model, s1_tokenizer, val_in_scope["text"].tolist(), label_names, device, max_length)
print(f"S1 {checkpoint} ({s1_source}) on {device}: {time.perf_counter() - start:.1f}s")

In [ ]:
# S2: no training. Embed the train split (cached after the first run) and match each query to its nearest example.
from router.systems import s2_embedding, s3_pairwise
from router.systems.registry import s2_support_set, s3_max_length

s2_cfg = data.load_config("s2_embedding")
start = time.perf_counter()
encoder = s2_embedding.load_encoder(s2_cfg["checkpoint"], device)
support = s2_support_set(encoder, s2_cfg["checkpoint"], s2_cfg["batch_size"])
s2_preds = s2_embedding.predict(encoder, support, val_in_scope["text"].tolist(), label_names, s2_cfg["batch_size"])
print(f"S2 {s2_cfg['checkpoint']} on {device}: {time.perf_counter() - start:.1f}s")

In [ ]:
# S3: load the saved seed-0 checkpoint if this machine has it; otherwise use the paraphrase
# model without any fine-tuning (fast mode), which still works because it already knows paraphrases.
s3_cfg = data.load_config("s3_pairwise")
s3_saved = data.PROJECT_ROOT / "checkpoints" / "s3" / s3_pairwise.system_tag(s3_cfg["checkpoint"]) / "seed0"
start = time.perf_counter()
if s3_saved.exists():
    s3_model, s3_tokenizer = s3_pairwise.load(s3_saved, device)
    s3_source = "saved seed-0 checkpoint"
else:
    s3_model, s3_tokenizer = s3_pairwise.load_base(s3_cfg["checkpoint"])
    s3_model.to(device)
    s3_source = "paraphrase model without fine-tuning (no checkpoint on this machine)"
s3_preds = s3_pairwise.predict(s3_model, s3_tokenizer, encoder, support, val_in_scope["text"].tolist(), label_names, s3_cfg, device, s3_max_length(s3_cfg))
print(f"S3 {s3_cfg['checkpoint']} ({s3_source}) on {device}: {time.perf_counter() - start:.1f}s")

In [ ]:
# Why S3 decides what it decides: each prediction comes with the labelled example it matched.
pd.DataFrame({
    "query": val_in_scope["text"].head(6).values,
    "true intent": val_in_scope["label"].head(6).values,
    "S3 intent": s3_preds.pred_label[:6],
    "pair probability": s3_preds.confidence[:6].round(3),
    "matched example": s3_preds.matched_example[:6],
})

## 4. Evaluation

E1 scores in-scope intent classification with accuracy, precision, recall and F1 (`src/router/metrics.py`, built on scikit-learn). Macro averages are primary because each of the 77 intents should count equally. Weighted averages are shown next to them.

**These are validation numbers.** The test split is not touched until every choice is frozen and the hypotheses are pre-registered (M5).

First, the numbers computed in this run (fast mode):

In [ ]:
from router.metrics import classification_metrics

METRIC_ORDER = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_precision", "weighted_recall", "weighted_f1"]
fast_rows = {
    "S0 (this run)": classification_metrics(val_in_scope["label"], s0_preds.pred_label, labels=label_names),
    f"S1 ({s1_source})": classification_metrics(val_in_scope["label"], s1_preds.pred_label, labels=label_names),
    "S2 (this run)": classification_metrics(val_in_scope["label"], s2_preds.pred_label, labels=label_names),
    f"S3 ({s3_source})": classification_metrics(val_in_scope["label"], s3_preds.pred_label, labels=label_names),
}
(pd.DataFrame(fast_rows).T[METRIC_ORDER] * 100).round(2)

**Reported results**: validation, mean ± standard deviation over five seeds, produced by `scripts/evaluate.py` (`results/e1_val.json`). S0 and S2 are deterministic, so their standard deviation is 0 (S2 runs once). Both S2 candidates are listed; S2 uses all-MiniLM-L6-v2.

About a quarter of validation queries have a near-paraphrase in train (cosine ≥ 0.95, `results/near_duplicates.md`), which favours nearest-neighbour systems like S2. Test has a similar rate, so validation remains a fair guide.

In [ ]:
with open(data.RESULTS_DIR / "e1_val.json") as f:
    e1 = json.load(f)

reported = {
    tag: {k: f"{100 * r['summary'][k]['mean']:.2f} ± {100 * r['summary'][k]['std']:.2f}" for k in METRIC_ORDER}
    for tag, r in e1["systems"].items()
}
pd.DataFrame(reported).T

In [ ]:
# Where S1 struggles: the 10 intents with the lowest F1 and the most frequent confusions (validation, all seeds).
s1_tag = s1_encoder.system_tag(checkpoint)
worst = pd.DataFrame(e1["systems"][s1_tag]["worst_intents"]).set_index("intent").round(3)
confused = pd.DataFrame(e1["systems"][s1_tag]["top_confused_pairs"])
display(worst)
display(confused)

### 4b. Out-of-scope detection (E3)

A router must also notice queries it should not answer at all. Each system's out-of-scope score (1 minus its confidence) is checked on the mixed validation set: 1,000 Banking77 queries plus 350 CLINC150 out-of-scope queries (`scripts/evaluate_oos.py`).

- **AUROC:** the chance that a random out-of-scope query scores higher than a random in-scope one. Higher is better; 50 is a coin flip.
- **FPR at 95% TPR:** keep 95% of in-scope queries; the share of out-of-scope queries that are still let through. Lower is better.

**Format control.** Every CLINC150 query is lowercase with no punctuation, while most Banking77 queries are capitalised and punctuated, so formatting alone almost separates the two (the surface-cue table below). Every query, in-scope and out-of-scope alike, is therefore lowercased and stripped of punctuation before scoring (`router.preprocess.format_control`). These format-controlled numbers are the primary ones. The models themselves are unchanged.

First, this run (fast mode):

In [ ]:
# E3 in this run: predict the mixed validation set (in-scope + out-of-scope) on format-controlled text.
from router.metrics import fpr_at_tpr, oos_auroc
from router.preprocess import format_control

val_mixed = data.load_eval_set("val", with_oos=True)
controlled_texts = val_mixed["text"].map(format_control).tolist()
is_oos = val_mixed["is_oos"].to_numpy(dtype=bool)

start = time.perf_counter()
fast_oos = {
    "S0 (this run)": s0_tfidf.predict(s0_model, controlled_texts, label_names),
    f"S1 ({s1_source})": s1_encoder.predict(s1_model, s1_tokenizer, controlled_texts, label_names, device, max_length),
    "S2 (this run)": s2_embedding.predict(encoder, support, controlled_texts, label_names, s2_cfg["batch_size"]),
    f"S3 ({s3_source})": s3_pairwise.predict(s3_model, s3_tokenizer, encoder, support, controlled_texts, label_names, s3_cfg, device, s3_max_length(s3_cfg)),
}
print(f"{len(controlled_texts)} queries ({is_oos.sum()} out of scope) scored by four systems in {time.perf_counter() - start:.1f}s")
pd.DataFrame({
    name: {"AUROC": 100 * oos_auroc(is_oos, p.oos_score), "FPR at 95% TPR": 100 * fpr_at_tpr(is_oos, p.oos_score)["fpr"]}
    for name, p in fast_oos.items()
}).T.round(2)

**Reported results**: validation, format-controlled text, mean ± standard deviation over seeds (`results/e3_val_fmt.json`), with the raw-text AUROC next to it. The comparisons use a paired bootstrap (1,000 resamples of the same queries for both systems): when the 95% interval excludes 0, the difference is unlikely to be chance. RQ2 asks whether S3 beats S1 here (first row).

In [ ]:
with open(data.RESULTS_DIR / "e3_val_fmt.json") as f:
    e3 = json.load(f)
with open(data.RESULTS_DIR / "e3_val.json") as f:
    e3_raw = json.load(f)


def mean_pm_std(summary: dict, key: str) -> str:
    return f"{100 * summary[key]['mean']:.2f} ± {100 * summary[key]['std']:.2f}"


display(pd.DataFrame({
    tag: {
        "AUROC": mean_pm_std(r["summary"], "auroc"),
        "FPR at 95% TPR": mean_pm_std(r["summary"], "fpr_at_tpr"),
        # S4 (the LLM) was only run on format-controlled text, to halve the API cost.
        "AUROC on raw text": mean_pm_std(e3_raw["systems"][tag]["summary"], "auroc") if tag in e3_raw["systems"] else "not run",
    }
    for tag, r in e3["systems"].items()
}).T)
display(pd.DataFrame([
    {"A": c["a"], "B": c["b"], "AUROC A - B (points)": round(100 * c["diff"], 2),
     "95% CI": f"[{100 * c['ci_low']:+.2f}, {100 * c['ci_high']:+.2f}]"}
    for c in e3["comparisons"]
]))

In [ ]:
# Why the control is needed: each formatting rule on its own, with no model (AUROC; 50 = no signal).
pd.DataFrame({
    "raw text": {c["cue"]: round(100 * c["auroc"], 1) for c in e3_raw["surface_cues"]},
    "format-controlled text": {c["cue"]: round(100 * c["auroc"], 1) for c in e3["surface_cues"]},
})

The control also changes in-scope accuracy (`results/e1_val_fmt.json`), which doubles as a check on lowercase, unpunctuated input, the way people often type in a chat.

In [ ]:
with open(data.RESULTS_DIR / "e1_val_fmt.json") as f:
    e1_fmt = json.load(f)
pd.DataFrame({
    tag: {"accuracy, raw text": mean_pm_std(e1["systems"][tag]["summary"], "accuracy") if tag in e1["systems"] else "not run",
          "accuracy, format-controlled": mean_pm_std(r["summary"], "accuracy")}
    for tag, r in e1_fmt["systems"].items()
}).T

In [ ]:
# ROC curves (seed 0 of each system), drawn by scripts/evaluate_oos.py.
from IPython.display import Image

Image(filename=str(data.PROJECT_ROOT / e3["figure"]), width=560)

In [ ]:
# The out-of-scope queries S3 accepted most confidently (seed 0): the misses to read first.
pd.DataFrame(e3["most_confidently_accepted"]).round(3)

### 4c. Label noise (E2)

Banking77 is known to contain labelling mistakes (Ying and Thomas, 2022). E2 asks whether they change this project's conclusions (RQ4: is the system ranking stable?).

- **Finding them:** cleanlab's confident learning (Northcutt et al., 2021). A separate model (frozen MPNet sentence embeddings plus logistic regression, not one of the compared systems) gives every train and validation query class probabilities from 5-fold cross-validation, so no query is scored by a model that saw its label. cleanlab flags the queries whose given label disagrees with those probabilities by more than the estimated noise rates allow (`scripts/find_label_issues.py`).
- **Cleaned** means the flagged queries are removed, never relabelled. Every system is retrained without the flagged train queries, with every other choice frozen, and scored with and without the flagged validation queries (`scripts/evaluate_label_noise.py`).
- **Flagged is not the same as wrong.** The owner hand-checks a random 100 flagged queries, blind to which label the dataset gave.

First, this run: re-derive the flags from the saved cross-validated probabilities (about a second).

In [ ]:
# E2 in this run: cleanlab on the saved out-of-fold probabilities gives the same flags as the committed list.
import numpy as np

from router import label_noise

e2_cfg = data.load_config("e2_label_noise")
saved = np.load(data.PROCESSED_DIR / label_noise.PRED_PROBS_FILE)
ranked = label_noise.flag_issues(saved["label_id"], saved["pred_probs"], e2_cfg["cleanlab"]["filter_by"], e2_cfg["cleanlab"]["rank_by"], e2_cfg["cleanlab"]["n_jobs"])
flagged = label_noise.load_flagged()
same = len(ranked) == len(flagged) and bool((saved["source"][ranked] == flagged["source"].to_numpy()).all()) \
    and bool((saved["orig_index"][ranked] == flagged["orig_index"].to_numpy()).all())
print(f"{len(ranked)} of {len(saved['label_id'])} queries flagged "
      f"({(flagged['split'] == 'train').sum()} train, {(flagged['split'] == 'val').sum()} validation); "
      f"identical to data/processed/banking77_flagged.csv: {same}")
flagged.head(8)[["split", "text", "given_label", "suggested_label", "given_label_prob"]]

**Reported results** (`results/e2_val.json`): in-scope macro-F1 on validation (raw text), mean ± standard deviation over seeds, in four cells: trained on original or cleaned labels, scored on all validation queries or with the flagged ones removed. The training effect uses a paired bootstrap stratified by intent (1,000 resamples).

In [ ]:
with open(data.RESULTS_DIR / "e2_val.json") as f:
    e2 = json.load(f)
e2_cells = list(e2["rankings"])
display(pd.DataFrame({name: {cell: mean_pm_std(r["cells"][cell], "macro_f1") for cell in e2_cells} for name, r in e2["systems"].items()}).T)
print("Ranking stable in all four cells (RQ4):", e2["ranking_stable"])
for cell, order in e2["rankings"].items():
    print(f"  {cell}: {' > '.join(order)}")
display(pd.DataFrame([
    {"system": name, "cleaned - original macro-F1 (points)": round(100 * r["training_effect"]["diff"], 2),
     "95% CI": f"[{100 * r['training_effect']['ci_low']:+.2f}, {100 * r['training_effect']['ci_high']:+.2f}]"}
    for name, r in e2["systems"].items()
]))

In [ ]:
# S4 is not retrained (it has no training data). Its error rate on the mixed set is E6's matched risk:
# how much of it comes from the flagged queries? (format-controlled text, cached replies, no API calls)
display(pd.DataFrame({
    s["name"]: {"error rate, all (%)": round(100 * s["all"]["error_rate"], 2),
                "error rate, flagged removed (%)": round(100 * s["flagged removed"]["error_rate"], 2)}
    for s in e2["s4_evaluation_effect"]
}).T)
audit = e2["audit"]
if audit and audit["complete"]:
    low, high = audit["real_error_ci95"]
    print(f"Audit: {audit['real_error']} of {audit['n_items']} flagged queries are real label errors "
          f"(95% interval {100 * low:.0f} to {100 * high:.0f}%); ambiguous {audit['ambiguous']}, false flags {audit['false_flag']}.")
else:
    print("Audit: not finished yet" + (f" ({audit['n_done']} of {audit['n_items']} items done)." if audit else "."))

## 5. Router and cost (E6, the headline result)

The router answers a query itself (HANDLE) when its confidence is at or above a threshold, and sends the rest to the LLM (ESCALATE). The LLM is S4: Claude Sonnet 4.6 on Amazon Bedrock, the model the assistant runs today, given the 77 intent names zero-shot at temperature 0.

- **Matched risk:** S4's error rate on the same queries. Each router's threshold is the one that handles the most queries while being wrong on them no more often than S4 is.
- **Coverage:** the share of queries the router handles, so the share of LLM calls avoided.
- **Cost per query** = (1 - coverage) x S4's measured cost per call (the local router's own cost is taken as zero; its latency is shown below).

Everything uses the format-controlled validation set (in-scope plus out-of-scope). S4's replies are cached in `results/s4_cache/`, so this notebook never calls the API.

First, this run (fast mode, with the systems in memory and no temperature scaling):

In [ ]:
import numpy as np

from router.metrics import aurc, coverage_at_risk, risk_coverage_curve
from router.systems.base import load_predictions

# Cached S4 replies (no API call), for both variants. The one with the lower error sets the matched risk.
s4_errors = {}
for tag in data.load_config("e6_router")["llm_systems"]:
    s4_preds, _ = load_predictions(tag, 0, "val_fmt")
    assert s4_preds["text"].tolist() == controlled_texts  # same queries, same order
    s4_errors[tag] = float((s4_preds["pred_label"] != s4_preds["label"]).mean())
    print(f"{tag}: error rate {100 * s4_errors[tag]:.2f}%")
s4_error = min(s4_errors.values())
true_labels = val_mixed["label"].to_numpy()
print(f"Matched risk on these {len(true_labels)} queries: {100 * s4_error:.2f}%")

fast_rows = {}
for name, p in fast_oos.items():
    correct = np.asarray(p.pred_label) == true_labels  # an out-of-scope query is never correct for S0-S3
    matched = coverage_at_risk(correct, p.confidence, s4_error)
    fast_rows[name] = {
        "coverage at matched risk (%)": 100 * matched["coverage"],
        "AURC (%)": 100 * aurc(risk_coverage_curve(correct, p.confidence)),
    }
pd.DataFrame(fast_rows).T.round(2)

**Reported results** (`scripts/evaluate_router.py`, `results/e6_val_fmt.json`): mean ± standard deviation over seeds, with S0 and S1 temperature-scaled on validation. "OOS handled" counts out-of-scope queries a router answered itself, the dangerous error. The thresholds and temperatures are frozen in `results/policy_v1.json` for the single test run.

In [ ]:
with open(data.RESULTS_DIR / "e6_val_fmt.json") as f:
    e6 = json.load(f)
# Both S4 variants; the one with the lower error rate sets the matched risk (rule fixed before E6 ran).
display(pd.DataFrame({
    tag: {"error rate (%)": round(100 * v["error_rate"], 2), "in-scope accuracy (%)": round(100 * v["in_scope_accuracy"], 2),
          "OOS recall (%)": round(100 * v["oos_recall"], 2), "US$ per 1,000 queries": round(1000 * v["cost_per_call_usd"], 3)}
    for tag, v in e6["s4_variants"].items()
}).T)
s4 = e6["s4"]
print(f"Matched risk set by {e6['llm_system']}: error rate {100 * s4['error_rate']:.2f}%, "
      f"US${1000 * s4['cost_per_call_usd']:.3f} per 1,000 queries if every query goes to the LLM")
display(pd.DataFrame({
    tag: {
        "coverage at matched risk (%)": mean_pm_std(r["summary"], "coverage"),
        "selective risk (%)": mean_pm_std(r["summary"], "selective_risk"),
        "OOS handled": f"{r['summary']['oos_handled']['mean']:.1f}",
        "cost per 1,000 queries (US$)": f"{r['summary']['cost_per_1000_usd']['mean']:.3f}",
        "saving vs LLM only (%)": mean_pm_std(r["summary"], "saving_vs_llm"),
    }
    for tag, r in e6["systems"].items()
}).T)
print("Best router (highest mean coverage at matched risk):", e6["best_router"])
pd.DataFrame([
    {"A": c["a"], "B": c["b"], "coverage A - B (points)": round(100 * c["diff"], 2),
     "95% CI": f"[{100 * c['ci_low']:+.2f}, {100 * c['ci_high']:+.2f}]"}
    for c in e6["comparisons"]
])

In [ ]:
# Risk-coverage curves (seed 0); dots mark each router's matched-risk operating point.
Image(filename=str(data.PROJECT_ROOT / e6["figure"]), width=560)

In [ ]:
# Latency per query on CPU (batch size 1, warm model), and S4's API latency for comparison.
latency_rows = {tag: {"p50 (ms)": round(v["p50_ms"], 1), "p95 (ms)": round(v["p95_ms"], 1)} for tag, v in e6["latency"]["systems"].items()}
latency_rows["S4 API (from Sydney)"] = {"p50 (ms)": round(s4["api_latency_ms"]["p50"]), "p95 (ms)": round(s4["api_latency_ms"]["p95"])}
print(e6["latency"]["machine"])
pd.DataFrame(latency_rows).T

## 6. Limitations

Written in M6, once the results are in.

In [ ]:
print(f"Notebook finished in {(time.perf_counter() - NOTEBOOK_START) / 60:.1f} min (FAST = {FAST}).")